# Offline with Parquet

seqout publishes its tables as Parquet files. The Parquet backend reads them
with DuckDB and sends no request to the API, which suits offline work, large
batch jobs, and queries the API does not expose.

The trade-off is coverage and speed. A Parquet query filters by reading the
file, so a query over the remote URL is slow for the large tables, and a few
API features have no Parquet equivalent. Both are covered below.

In [1]:
from seqout import connect

pq = connect(backend="parquet")

## The data source

The source is a URL or a local directory. The public dump is at
`https://seqout.org/data`, which is also the default.

In [2]:
pq.set_source("https://seqout.org/data")

For repeated work, download the files once and read them from disk. The
command below writes about 15 GB, most of it the run download links table.

```bash
seqout parquet download /data/seqout
```

```python
pq.set_source("/data/seqout")
```

## The same methods as the API backend

Both clients carry the same method names and return the same models, so code
written against one runs against the other.

In [3]:
study = pq.fetch_study("GSE169470")

print(study.title)
print(study.num_experiments, "experiments,", study.num_samples, "samples")
print("organisms:", study.organisms)

RNA-sequencing Analysis (RNA-seq)，Genome-wide Maps of Chromatin State (ChIP-seq) and Assay for Transposase Accessible Chromatin with High-throughput Sequencing (ATAC-seq) in BMDMs or Raw 264.7 cell lines.
47 experiments, 47 samples
organisms: ['Mus musculus']


`get` works here too. It crosses the GEO/SRA link through the `aliases` column
of the unified metadata table rather than through the API's cross-reference
endpoint.

In [4]:
d = pq.get("GSE169470")
print("geo =", d.geo, "| sra =", d.sra)

geo = GSE169470 | sra = SRP311850


## Your own SQL

`execute_query` runs SQL through DuckDB. The backend replaces each table name
in the query with a read of its Parquet file. Because it finds table names by
text, do not reuse a table name as a column alias — use a distinct alias such
as `AS n`. The result has `.df()` for pandas.

In [5]:
pq.execute_query(
    "SELECT source, COUNT(*) AS n FROM unified_metadata GROUP BY source ORDER BY n DESC"
).df()

,source,n
0,sra,684870
1,geo,273727
2,ena,243708
3,ae,21008


This is the main reason to use the backend directly: aggregate questions that
no API endpoint answers.

In [6]:
pq.execute_query(
    """
    SELECT
        dominant_scientific_name AS organism,
        COUNT(*) AS studies,
        ROUND(AVG(n_samples), 1) AS mean_samples
    FROM unified_metadata
    WHERE is_single_cell AND dominant_scientific_name IS NOT NULL
    GROUP BY organism
    ORDER BY studies DESC
    LIMIT 10
    """
).df()

,organism,studies,mean_samples
0,Mus musculus,15478,70.3
1,Homo sapiens,11091,81.5
2,Danio rerio,529,48.7
3,Rattus norvegicus,320,21.3
4,Drosophila melanogaster,300,319.0
5,Sus scrofa,204,15.3
6,Macaca mulatta,165,188.3
7,Gallus gallus,153,106.5
8,Arabidopsis thaliana,131,57.1
9,Bos taurus,94,24.3


## What the Parquet backend does not do

Some fields have no table in the dump. Reading them raises `SeqoutError` that
names the field and says to use the API backend.

In [7]:
from seqout.exception import SeqoutError

for field in ("links", "enriched"):
    try:
        getattr(pq.get("GSE169470"), field)
    except SeqoutError as e:
        print(f"{field}: {e}")

links: cross_references is not available on the parquet backend (no such table in the dump). Use the API backend for it: connect() instead of connect('parquet').
enriched: enriched_metadata is not available on the parquet backend (no such table in the dump). Use the API backend for it: connect() instead of connect('parquet').


`classify` and `summaries` are also API-only. Two further differences are
worth knowing:

- `author` on Parquet matches GEO contributor names by substring. The API
  version searches publication author lists across every source, so the two
  return different sets.
- `run_download_links.parquet` is about 11.6 GB and is not sorted by study, so
  a run lookup over the remote URL reads a large part of the file. Use a local
  copy for that work.